# Frame Extraction - Videolardan Kare Çıkarma
**Bitirme Tezi:** Taralı Alanlarda Kaynak Yapan Araçların Tespiti

Bu notebook Drive'daki videolardan belirli aralıklarla frame çıkarır ve Roboflow'a yüklemeye hazır hale getirir.

## 1. Drive'ı Bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Video Klasörünü Ayarla
Aşağıdaki yolu kendi Drive klasörüne göre düzenle.

In [ ]:
import os

# Drive'daki video klasoru yolu
# Drive'i bagladiktan sonra sol panelden klasor yolunu kontrol et
VIDEO_DIR = "/content/drive/MyDrive/İstanbul Trafiği Kayıt"  # <- kendi yolunu yaz

# Cikti klasoru (frame'ler buraya kaydedilecek)
OUTPUT_DIR = "/content/drive/MyDrive/tez_frames"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Videolari listele
videos = [f for f in os.listdir(VIDEO_DIR) if f.lower().endswith(('.mov', '.mp4'))]
print(f"Bulunan videolar ({len(videos)}):")
for v in sorted(videos):
    size_mb = os.path.getsize(os.path.join(VIDEO_DIR, v)) / (1024*1024)
    print(f"  {v} ({size_mb:.0f} MB)")

## 3. Frame Çıkarma
Her 2 saniyede 1 kare çıkarır. Toplam ~900 frame elde edilir (30 dk video için).

In [ ]:
import cv2
from tqdm import tqdm

FRAME_INTERVAL_SEC = 2  # kac saniyede bir kare cikarilacak

total_extracted = 0

for video_name in sorted(videos):
    video_path = os.path.join(VIDEO_DIR, video_name)
    cam_name = os.path.splitext(video_name)[0]  # cam1, cam2, cam3

    # Her kamera icin ayri klasor
    cam_output = os.path.join(OUTPUT_DIR, cam_name)
    os.makedirs(cam_output, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps if fps > 0 else 0
    frame_step = int(fps * FRAME_INTERVAL_SEC)

    print(f"\n--- {video_name} ---")
    print(f"  FPS: {fps:.1f} | Sure: {duration_sec:.0f}s | Toplam frame: {total_frames}")
    print(f"  Her {FRAME_INTERVAL_SEC}s'de 1 kare = ~{int(duration_sec/FRAME_INTERVAL_SEC)} frame cikarilacak")

    count = 0
    frame_num = 0

    pbar = tqdm(total=total_frames, desc=cam_name)
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_num % frame_step == 0:
            timestamp = frame_num / fps
            filename = f"{cam_name}_frame_{frame_num:06d}_t{timestamp:.1f}s.jpg"
            filepath = os.path.join(cam_output, filename)
            cv2.imwrite(filepath, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            count += 1

        frame_num += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    total_extracted += count
    print(f"  {count} frame cikarildi -> {cam_output}")

print(f"\n=== TOPLAM: {total_extracted} frame cikarildi ===")

## 4. Örnek Frame'leri Görüntüle
Çıkarılan frame'lerden birkaçını kontrol et.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for idx, cam_name in enumerate(['cam1', 'cam2', 'cam3', 'cam4', 'cam5']):
    cam_dir = os.path.join(OUTPUT_DIR, cam_name)
    if not os.path.exists(cam_dir):
        axes[idx].set_title(f"{cam_name} - YOK")
        axes[idx].axis('off')
        continue
    frames = sorted(os.listdir(cam_dir))
    if not frames:
        axes[idx].set_title(f"{cam_name} - BOS")
        axes[idx].axis('off')
        continue
    # Ortadan bir frame sec
    mid = len(frames) // 2
    img = cv2.imread(os.path.join(cam_dir, frames[mid]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[idx].imshow(img)
    axes[idx].set_title(f"{cam_name} - {frames[mid]}")
    axes[idx].axis('off')

plt.suptitle("5 Kameradan Ornek Frame", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Roboflow'a Yükleme

### Manuel Yükleme (Kolay yol):
1. Drive'da `tez_frames` klasörünü aç
2. `cam1`, `cam2`, `cam3` klasörlerindeki JPG'leri indir (ZIP olarak)
3. Roboflow'da yeni proje oluştur:
   - Proje adı: `istanbul-hatched-area`
   - Annotation Group: `Object Detection`
   - Sınıflar: `car`, `truck`, `bus`, `motorcycle`
4. Görselleri yükle ve etiketlemeye başla

### API ile Yükleme (Hızlı yol):

In [ ]:
# Roboflow API ile yukleme (opsiyonel)
# Once: pip install roboflow

# !pip install roboflow

# from roboflow import Roboflow
# rf = Roboflow(api_key="SENIN_API_KEY")  # <- Roboflow Settings > API Key
# project = rf.workspace().project("istanbul-hatched-area")
#
# for cam_name in ['cam1', 'cam2', 'cam3']:
#     cam_dir = os.path.join(OUTPUT_DIR, cam_name)
#     for img_file in sorted(os.listdir(cam_dir)):
#         img_path = os.path.join(cam_dir, img_file)
#         project.upload(img_path, split="train")
#         print(f"Yuklendi: {img_file}")

## Sonraki Adım

Frame'ler Roboflow'a yüklendikten sonra:
1. Her görselde araçların etrafına **bounding box** çiz
2. Sınıf olarak `car`, `truck`, `bus`, `motorcycle` seç
3. Roboflow'un **auto-label** özelliğini kullan (ilk 50-100 elle etiketle, sonra otomatik)
4. Etiketleme bitince **YOLOv8 formatında export** et
5. `03_vehicle_detection_finetuning.ipynb` notebook'unu çalıştır

### Özet
- **5 kamera**: cam1, cam2, cam3, cam4, cam5
- **Frame aralığı**: Her 2 saniyede 1 kare
- **Tahmini toplam**: ~1000-1500 frame (video sürelerine bağlı)